# GĐ 2: Full Pipeline Retrain on Clean Data

**Steps:**
1. Setup + Data preparation (dedup)
2. Train embedding (contrastive, DeBERTa)
3. Build FAISS index
4. Train ABSA experiments (A: no-retrieval, B: inv-freq, C: sqrt)
5. Evaluate all experiments

**Kaggle setup:** T4 GPU, ~3-4 hours total

**Input datasets:**
- `lcminhc/semeval-absa-restaurant` (XML files)

## 0. Setup

In [ ]:
!pip install -q transformers faiss-cpu lxml scikit-learn pyyaml

In [ ]:
import os
import sys
import json
import shutil

# Clone the repo
!git clone https://github.com/lucminhduc3108/Retrieval-ABSA.git /kaggle/working/repo
os.chdir('/kaggle/working/repo')
sys.path.insert(0, '/kaggle/working/repo')

print('Working dir:', os.getcwd())
print('Files:', os.listdir('.'))

In [ ]:
# Symlink SemEval dataset from Kaggle input
KAGGLE_INPUT = '/kaggle/input/semeval-absa-restaurant'

# Create expected directory structure
os.makedirs('SemEval-Dataset/SemEval 2015 Task 12/ABSA15_RestaurantsTrain', exist_ok=True)
os.makedirs('SemEval-Dataset/SemEval 2015 Task 12/Gold Annotation', exist_ok=True)
os.makedirs('SemEval-Dataset/SemEval 2016 Task 5/Restaurant Training', exist_ok=True)
os.makedirs('SemEval-Dataset/SemEval 2016 Task 5/Phase B/Gold Annotation/Restaurant', exist_ok=True)

# Copy XML files to expected locations
shutil.copy(f'{KAGGLE_INPUT}/semeval15_restaurants_train.xml',
            'SemEval-Dataset/SemEval 2015 Task 12/ABSA15_RestaurantsTrain/ABSA-15_Restaurants_Train_Final.xml')
shutil.copy(f'{KAGGLE_INPUT}/semeval15_restaurants_test.xml',
            'SemEval-Dataset/SemEval 2015 Task 12/Gold Annotation/ABSA15_Restaurants_Test.xml')
shutil.copy(f'{KAGGLE_INPUT}/semeval16_restaurants_train.xml',
            'SemEval-Dataset/SemEval 2016 Task 5/Restaurant Training/ABSA16_Restaurants_Train_SB1_v2.xml')
shutil.copy(f'{KAGGLE_INPUT}/semeval16_restaurants_test.xml',
            'SemEval-Dataset/SemEval 2016 Task 5/Phase B/Gold Annotation/Restaurant/EN_REST_SB1_TEST.xml.gold')

print('XML files in place.')

## 1. Data Preparation (with dedup)

In [ ]:
!python scripts/01_prepare_data.py
print('\n--- Verifying dedup ---')
!python scripts/analyze_duplicates.py

## 2. Train Contrastive Embedding

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# Train embedding with batch_size=32 (upgrade from 16)
# Triplets are smaller now (1,537 vs 4,158) so training is faster

import yaml

# Override embedding config for this session
emb_cfg = {
    'model_name': 'microsoft/deberta-v3-base',
    'proj_dim': 256,
    'tau': 0.07,
    'batch_size': 32,
    'epochs': 15,
    'lr': 2.0e-5,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'max_seq_length': 128,
    'grad_clip': 1.0,
    'patience': 5,
    'seed': 42,
    'triplets_path': 'data/processed/contrastive_triplets.jsonl',
    'val_ratio': 0.1,
    'ckpt_dir': 'checkpoints/embedding',
    'log_path': 'logs/embedding_training.jsonl',
}

with open('configs/embedding_v2.yaml', 'w') as f:
    yaml.dump(emb_cfg, f)

print('Embedding config (v2):')
print(f'  batch_size: {emb_cfg["batch_size"]} (was 16)')
print(f'  epochs: {emb_cfg["epochs"]} (was 10)')
print(f'  patience: {emb_cfg["patience"]} (was 3)')

In [ ]:
!python scripts/02_train_embedding.py --config configs/embedding_v2.yaml

In [ ]:
# Check embedding training log
if os.path.exists('logs/embedding_training.jsonl'):
    with open('logs/embedding_training.jsonl') as f:
        for line in f:
            rec = json.loads(line)
            print(f"Epoch {rec.get('epoch', '?')}: loss={rec.get('train_loss', 0):.4f}, "
                  f"R@1={rec.get('recall@1', 0):.4f}, R@3={rec.get('recall@3', 0):.4f}, "
                  f"R@5={rec.get('recall@5', 0):.4f}")

## 3. Build FAISS Index

In [ ]:
!python scripts/03_build_index.py \
    --embedding_ckpt checkpoints/embedding/best.pt \
    --input data/processed/classification.jsonl \
    --bio_input data/processed/bio_tagging.jsonl \
    --out_dir indexes/

## 4. Train ABSA Experiments

### Experiment A: No-retrieval baseline

In [ ]:
!python scripts/04_train_absa.py \
    --config configs/absa.yaml \
    --no_retrieval \
    --grad_accum_steps 8 \
    --ckpt_path checkpoints/absa/exp_a.pt

### Experiment B: Retrieval + inverse-freq class weights

In [ ]:
# Clear GPU memory before next experiment
import gc
gc.collect()
torch.cuda.empty_cache()

!python scripts/04_train_absa.py \
    --config configs/absa_exp_b.yaml \
    --embedding_ckpt checkpoints/embedding/best.pt \
    --index_dir indexes/ \
    --grad_accum_steps 8 \
    --ckpt_path checkpoints/absa/exp_b.pt

### Experiment C: Retrieval + sqrt class weights

In [ ]:
gc.collect()
torch.cuda.empty_cache()

!python scripts/04_train_absa.py \
    --config configs/absa_exp_c.yaml \
    --embedding_ckpt checkpoints/embedding/best.pt \
    --index_dir indexes/ \
    --grad_accum_steps 8 \
    --ckpt_path checkpoints/absa/exp_c.pt

## 5. Evaluate All Experiments

In [ ]:
print('='*60)
print('EXPERIMENT A: No-retrieval baseline')
print('='*60)
!python scripts/05_evaluate.py \
    --config configs/absa.yaml \
    --checkpoint checkpoints/absa/exp_a.pt \
    --no_retrieval

In [ ]:
print('='*60)
print('EXPERIMENT B: Retrieval + inv-freq class weights')
print('='*60)
!python scripts/05_evaluate.py \
    --config configs/absa_exp_b.yaml \
    --checkpoint checkpoints/absa/exp_b.pt \
    --embedding_ckpt checkpoints/embedding/best.pt \
    --index_dir indexes/

In [ ]:
print('='*60)
print('EXPERIMENT C: Retrieval + sqrt class weights')
print('='*60)
!python scripts/05_evaluate.py \
    --config configs/absa_exp_c.yaml \
    --checkpoint checkpoints/absa/exp_c.pt \
    --embedding_ckpt checkpoints/embedding/best.pt \
    --index_dir indexes/

## 6. Results Summary

In [ ]:
# Collect and display results comparison table
import json

def read_last_eval(log_path):
    """Read eval metrics from training log (last epoch with val metrics)."""
    if not os.path.exists(log_path):
        return None
    last = None
    with open(log_path) as f:
        for line in f:
            rec = json.loads(line)
            if 'val_joint_f1' in rec:
                last = rec
    return last

print('\n' + '='*80)
print('RESULTS COMPARISON TABLE')
print('='*80)
print(f'{"Metric":<20} {"MVP (old)":<12} {"Exp A":<12} {"Exp B":<12} {"Exp C":<12}')
print('-'*68)

mvp = {'joint_f1': 0.6379, 'span_f1': 0.7088, 'sent_acc': 0.9079, 'sent_macro_f1': 0.7619}

# Note: actual results will be printed by evaluate.py above
# This cell is a template - fill in after running
print(f'{"Joint F1":<20} {mvp["joint_f1"]:<12.4f} {"?":<12} {"?":<12} {"?":<12}')
print(f'{"Span F1":<20} {mvp["span_f1"]:<12.4f} {"?":<12} {"?":<12} {"?":<12}')
print(f'{"Sent Acc":<20} {mvp["sent_acc"]:<12.4f} {"?":<12} {"?":<12} {"?":<12}')
print(f'{"Sent Macro F1":<20} {mvp["sent_macro_f1"]:<12.4f} {"?":<12} {"?":<12} {"?":<12}')
print('\nNote: MVP was trained on dirty data (4,161 train with 49.8% test leakage).')
print('Current experiments use clean data (1,707 train, 0.3% leakage).')
print('Direct comparison with MVP is NOT fair - clean data results are more reliable.')

## 7. Save Checkpoints as Kaggle Dataset

In [ ]:
# Save all checkpoints for future use
output_dir = '/kaggle/working/checkpoints_gd2'
os.makedirs(output_dir, exist_ok=True)

for name in ['embedding/best.pt', 'absa/exp_a.pt', 'absa/exp_b.pt', 'absa/exp_c.pt']:
    src = f'checkpoints/{name}'
    if os.path.exists(src):
        dst = os.path.join(output_dir, name.replace('/', '_'))
        shutil.copy(src, dst)
        size_mb = os.path.getsize(dst) / 1e6
        print(f'Saved {dst} ({size_mb:.1f} MB)')

# Also save the index
for f in ['train.faiss', 'train_vectors.npy', 'train_metadata.jsonl']:
    src = f'indexes/{f}'
    if os.path.exists(src):
        shutil.copy(src, os.path.join(output_dir, f))

# Save training logs
if os.path.exists('logs'):
    shutil.copytree('logs', os.path.join(output_dir, 'logs'), dirs_exist_ok=True)

print(f'\nAll outputs saved to {output_dir}')
print('Download as dataset or use Kaggle API to save.')